# 01 · GPU PyTorch — Train · Log · Serve

PyTorch MLP 를 **단일 GPU** 에서 학습하고, MLflow 의 `pytorch` flavor 로 logging 한 뒤 **GPU Model Serving endpoint** 로 배포합니다.

## 이 노트북에서 배우는 것

1. **GPU 학습**: `torch.cuda` device 설정, DataLoader → MLP → Adam 학습 루프
2. **`mlflow.pytorch` flavor**: state_dict 와 클래스 정의를 함께 logging, signature 추론
3. **UC 등록 + alias**: `models:/<name>@champion` URI
4. **GPU endpoint**: `workload_type=GPU_SMALL` (T4), `scale_to_zero_enabled=True`
5. **호출**: REST + Python SDK 로 endpoint 검증

## 사전 요구사항
- [`../01-mlflow-logging/00-setup.ipynb`](../01-mlflow-logging/00-setup.ipynb) 을 먼저 실행해 `customers` 테이블이 있어야 함
- **GPU 클러스터** (DBR ML 15.x 이상, 예: `g4dn.xlarge` 1× T4)
- Serving entitlement (workspace admin 이 enable)

In [ ]:
%pip install -q "mlflow>=2.20.0" "databricks-sdk>=0.30.0" "torch>=2.0"
%restart_python

In [ ]:
%run ./config

In [ ]:
import mlflow
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_path)

## Step 1. GPU 가용성 확인

`torch.cuda.is_available()` 가 `False` 면 클러스터가 GPU 가 아닙니다. 노트북을 멈추고 GPU 클러스터로 attach 하세요.

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU 가 보이지 않습니다. DBR ML + GPU 인스턴스(예: g4dn.xlarge) 클러스터로 변경하세요."

device = torch.device("cuda:0")
print(f"device       : {device}")
print(f"gpu          : {torch.cuda.get_device_name(0)}")
print(f"torch        : {torch.__version__}")
print(f"cuda runtime : {torch.version.cuda}")

## Step 2. 데이터 로드 → torch DataLoader

01 챕터에서 만든 `customers` 테이블을 그대로 사용. binary churn (0/1) target.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

pdf = spark.table(f"{catalog}.{schema}.customers").toPandas()

FEATURES = ["age", "tenure_months", "monthly_charges", "total_charges", "support_tickets"]
TARGET = "churned"

X_train, X_test, y_train, y_test = train_test_split(
    pdf[FEATURES].values, pdf[TARGET].values,
    test_size=0.2, stratify=pdf[TARGET], random_state=42,
)

# 표준화 — endpoint 호출 시에도 같은 통계가 필요하므로 train 통계를 저장
mean = X_train.mean(axis=0)
std = X_train.std(axis=0) + 1e-6
X_train_n = (X_train - mean) / std
X_test_n = (X_test - mean) / std

train_ds = TensorDataset(
    torch.tensor(X_train_n, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long),
)
test_ds = TensorDataset(
    torch.tensor(X_test_n, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long),
)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

print(f"train batches={len(train_loader)}, test batches={len(test_loader)}")

## Step 3. MLP 정의

입력 5 (feature) → hidden 64 → output 2 (logits). 작은 모델이지만 `mlflow.pytorch` flavor 의 logging 패턴을 그대로 보여 줍니다.

In [ ]:
import torch.nn as nn

class ChurnMLP(nn.Module):
    def __init__(self, in_dim: int = 5, hidden: int = 64, out_dim: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)

model = ChurnMLP().to(device)
print(model)

## Step 4. 학습 루프

- Optimizer: Adam (lr=1e-3)
- Loss: `CrossEntropyLoss`
- Epoch: 10 (5K row 모델이라 빠르게 수렴)
- MLflow autolog 대신 **수동 로깅**으로 각 epoch metric 캡처

In [ ]:
import torch.optim as optim

EPOCHS = 10
LR = 1e-3

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

with mlflow.start_run(run_name="churn-torch-gpu") as run:
    mlflow.log_params({"epochs": EPOCHS, "lr": LR, "hidden": 64, "device": "cuda:0"})

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
        train_loss = running_loss / len(train_ds)

        model.eval()
        correct = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).argmax(dim=1)
                correct += (pred == yb).sum().item()
        val_acc = correct / len(test_ds)

        mlflow.log_metrics({"train/loss": train_loss, "val/acc": val_acc}, step=epoch)
        print(f"epoch {epoch:>2d}  train_loss={train_loss:.4f}  val_acc={val_acc:.4f}")

    run_id = run.info.run_id
    print(f"\nrun_id = {run_id}")

## Step 5. MLflow `pytorch` flavor 로 logging + UC 등록

`mlflow.pytorch.log_model` 은 state_dict 와 모델 클래스 정의를 함께 logging 합니다. 클래스 정의를 endpoint 가 import 할 수 있도록 **`code_paths` 로 모듈을 첨부**하는 것이 일반적입니다. 본 노트북은 노트북 셀 안에 클래스가 정의돼 있으므로 `code_paths` 없이 pickle-by-reference 가 되도록 `__main__` 모듈에 두는 것을 피하기 위해 **임시 모듈 파일을 생성**합니다.

In [ ]:
# 클래스 정의를 별도 .py 로 떨어뜨려야 pickle 이 endpoint 에서 복원 가능
import os, textwrap

CODE_DIR = "/tmp/churn_torch_code"
os.makedirs(CODE_DIR, exist_ok=True)

with open(f"{CODE_DIR}/churn_model.py", "w") as f:
    f.write(textwrap.dedent('''
        import torch.nn as nn

        class ChurnMLP(nn.Module):
            def __init__(self, in_dim: int = 5, hidden: int = 64, out_dim: int = 2):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(in_dim, hidden),
                    nn.ReLU(),
                    nn.Linear(hidden, hidden),
                    nn.ReLU(),
                    nn.Linear(hidden, out_dim),
                )

            def forward(self, x):
                return self.net(x)
    ''').strip() + "\n")

print(f"✓ wrote {CODE_DIR}/churn_model.py")

In [ ]:
# 같은 정의를 import 해서 state_dict 옮기기 — pickle 호환성 보장
import sys
sys.path.insert(0, CODE_DIR)
from churn_model import ChurnMLP as ChurnMLPModule  # noqa: E402

export_model = ChurnMLPModule().to(device)
export_model.load_state_dict(model.state_dict())
export_model.eval()

In [ ]:
import numpy as np
from mlflow.models import infer_signature

# Signature 추론용 sample input (raw — 표준화 전)
sample_input = pdf[FEATURES].head(3).values.astype("float32")
# 추론용 wrapper 가 표준화까지 책임지도록 PyFunc 으로 logging
class ChurnPyfunc(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import torch, json
        import sys, os
        sys.path.insert(0, context.artifacts["code"])
        from churn_model import ChurnMLP

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = ChurnMLP()
        self.model.load_state_dict(torch.load(context.artifacts["state_dict"], map_location=self.device))
        self.model.to(self.device).eval()

        stats = json.load(open(context.artifacts["stats"]))
        self.mean = np.array(stats["mean"], dtype="float32")
        self.std = np.array(stats["std"], dtype="float32")

    def predict(self, context, model_input, params=None):
        import torch
        arr = np.asarray(model_input, dtype="float32")
        normed = (arr - self.mean) / self.std
        with torch.no_grad():
            logits = self.model(torch.from_numpy(normed).to(self.device))
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        return probs

# state_dict + 표준화 통계 + 코드 디렉터리를 artifacts 로 번들
import json
STATE_PATH = f"{CODE_DIR}/state_dict.pt"
STATS_PATH = f"{CODE_DIR}/stats.json"
torch.save(export_model.state_dict(), STATE_PATH)
json.dump({"mean": mean.tolist(), "std": std.tolist()}, open(STATS_PATH, "w"))

sample_output = np.array([0.12, 0.87, 0.34], dtype="float32")  # 임의 — shape 만 신호
signature = infer_signature(sample_input, sample_output)

with mlflow.start_run(run_id=run_id):
    info = mlflow.pyfunc.log_model(
        name="model",
        python_model=ChurnPyfunc(),
        artifacts={
            "state_dict": STATE_PATH,
            "stats": STATS_PATH,
            "code": CODE_DIR,
        },
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            f"torch=={torch.__version__.split('+')[0]}",
            "numpy",
        ],
        input_example=sample_input,
        signature=signature,
        registered_model_name=model_torch_gpu,
    )

print(f"✓ logged model: {info.model_uri}")
print(f"✓ registered  : {model_torch_gpu}  version={info.registered_model_version}")

## Step 6. Alias 부여 (`@champion`)

In [ ]:
from mlflow import MlflowClient
client = MlflowClient()
client.set_registered_model_alias(
    name=model_torch_gpu, alias="champion", version=info.registered_model_version,
)
print(f"✓ alias set: {model_torch_gpu}@champion → v{info.registered_model_version}")

## Step 7. 로드 검증 — endpoint 띄우기 전 PyFunc 동작 확인

endpoint 가 5~10분 걸리므로 먼저 로컬에서 모델을 다시 로드해 PyFunc 가 동작하는지 확인합니다. 동일 로직이 endpoint container 안에서 일어납니다.

In [ ]:
loaded = mlflow.pyfunc.load_model(f"models:/{model_torch_gpu}@champion")
probs = loaded.predict(sample_input)
print("sample probs (churn=1):", probs)

## Step 8. GPU Serving endpoint 생성

| 옵션 | 값 |
| --- | --- |
| `workload_type` | **`GPU_SMALL`** (T4 1장) — `GPU_MEDIUM`, `GPU_LARGE` 도 선택 가능 |
| `workload_size` | `Small` (1 replica) |
| `scale_to_zero_enabled` | `True` — idle 시 $0 |
| `entity_version` | 위에서 등록한 버전 |

첫 생성은 image build + GPU node provision 으로 **5~15분** 걸립니다.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

w = WorkspaceClient()

served = ServedEntityInput(
    entity_name=model_torch_gpu,
    entity_version=info.registered_model_version,
    workload_type="GPU_SMALL",
    workload_size="Small",
    scale_to_zero_enabled=True,
)
config = EndpointCoreConfigInput(served_entities=[served])

try:
    endpoint = w.serving_endpoints.create_and_wait(
        name=endpoint_torch_gpu, config=config, timeout=__import__("datetime").timedelta(minutes=30),
    )
    print(f"✓ endpoint ready: {endpoint.name} (state={endpoint.state.ready})")
except Exception as e:
    # 이미 존재하면 update 로 전환
    print(f"create failed ({e}); attempting update_config_and_wait")
    endpoint = w.serving_endpoints.update_config_and_wait(
        name=endpoint_torch_gpu, served_entities=[served],
        timeout=__import__("datetime").timedelta(minutes=30),
    )
    print(f"✓ endpoint updated: {endpoint.name}")

## Step 9. Endpoint 호출

`dataframe_records` 포맷으로 raw feature 를 그대로 보냅니다. 표준화는 endpoint 안의 PyFunc 가 책임집니다.

In [ ]:
records = pdf[FEATURES].head(5).to_dict(orient="records")
print("input:")
for r in records:
    print(" ", r)

resp = w.serving_endpoints.query(
    name=endpoint_torch_gpu,
    dataframe_records=records,
)
print("\noutput (churn=1 probability):")
print(resp.predictions)

## ➡️ 다음 단계

- GPU endpoint 의 cold start / scale-to-zero 동작을 보고 싶으면 5분 정도 호출하지 않은 뒤 다시 query 해 보세요. 첫 호출만 latency 가 길고 (~30초) 이후는 ms 단위.
- 다른 호출 방식(REST, Spark UDF, AI Gateway) 비교는 [`../03-model-serving/01-model-serving.ipynb`](../03-model-serving/01-model-serving.ipynb) 의 Step 6 이후를 참고하세요. CPU endpoint 와 호출 인터페이스는 동일합니다.
- 핸즈온 종료 시 [`../03-model-serving/99-cleanup.ipynb`](../03-model-serving/99-cleanup.ipynb) 를 실행해 endpoint 를 정리합니다. `99-cleanup` 은 `endpoint_torch_gpu` 도 함께 삭제합니다.